In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'Census', 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
indicator_name = 'Cost_6'
# estimate = 'ACS1'
estimate = 'ACS5'

In [ ]:
df_eth = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_H_race.csv')
                                , dtype = {'State FIPS': object})
df_int = pd.read_csv(os.path.join(path_agol, indicator_name, 'Inter', indicator_name + '_PUMA_' + estimate + '_H_costs.csv')
                                , dtype = {'State FIPS': object})

# df_p = df_p.drop('RT', axis = 1)
# df_h = df_h.drop('RT', axis = 1)

In [ ]:
df_eth.head()

In [ ]:
df_int.info()

In [ ]:
# Renters vs Owners
conditions = [
                ( (df_int['WGTP'] == 0) ),
                ((df_int['GRPIP'] == 0) & (df_int['OCPIP'] == 0)),
                ( (df_int['GRPIP'] == 0) & (df_int['OCPIP'] > 0) ),
                ( (df_int['OCPIP'] == 0) & (df_int['GRPIP'] > 0) )
            ]

choices = ['Housing data not available', 'N/A (GQ/vacant/not owned or being bought/occupied without rent payment/no household income)',
           'Owner', 'Renter']
df_int["housing_type"] = np.select(conditions, choices)


# Cost burden
conditions = [
                ( (df_int['WGTP'] == 0) ),
                ((df_int['GRPIP'] == 0) & (df_int['OCPIP'] == 0)),
                ( ((df_int['GRPIP'] == 0) & (df_int['OCPIP'] <= 30)) | ((df_int['OCPIP'] == 0) & (df_int['GRPIP'] <= 30)) ),
                ( ((df_int['GRPIP'] > 30) & (df_int['GRPIP'] <= 50)) | ((df_int['OCPIP'] > 30) & (df_int['OCPIP'] <= 50)) ),
                ( (df_int['GRPIP']  > 50) | (df_int['OCPIP']  > 50) )
            ]

choices = ['Housing data not available', 'N/A (GQ/vacant/not owned or being bought/occupied without rent payment/no household income)',
           'Cost burden <=30%', 'Cost burden >30% to <=50%', 'Cost burden >50%']
df_int["housing_burden"] = np.select(conditions, choices)
df_int_merge = df_int[['State FIPS', 'PUMA', 'PUMA NAME', 'SERIALNO', 'year', 'housing_type', 'housing_burden']]

df_acs = df_eth.merge(df_int_merge, on = ['State FIPS', 'PUMA', 'PUMA NAME', 'SERIALNO', 'year'])


df_acs.head()

In [ ]:
# report_theme = 'Vibrant and Inclusive Places'
# sp_folder_out = 'Development\\Housing Cost'

# # Set file path for exporting
# path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name)
# path_out_csv  = os.path.join(path_agol, indicator_name)

# print('Excel files exported here: ' + path_out_xlsx)
# df_p2.to_excel(os.path.join(path_out_xlsx, 'QC Cost_6 Merge P and H.xlsx'), index = False)

# print('')
# print("Successfully exported")

In [ ]:
df_fips1 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'CountyFIPS' 
                                , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips2 = pd.read_excel(os.path.join(path_config0, 'Area Codes.xlsx'), sheet_name = 'PUMAcodes' 
                                , dtype = {'STATEFP': object, 'COUNTYFP': object})

df_fips1 = df_fips1[df_fips1['State FIPS'].isin(['06', '08'])]
df_fips2 = df_fips2[df_fips2['STATEFP'   ].isin(['06', '08'])]

df_fips2['PUMA5CE'] = df_fips2['PUMA5CE'].astype(str).apply('{:0>5}'.format)
df_fips2 = df_fips2[['STATEFP', 'COUNTYFP', 'PUMA5CE', 'Years']].drop_duplicates()
df_fips2 = df_fips2.rename(columns = {'STATEFP':'State FIPS', 'COUNTYFP':'County FIPS', 'PUMA5CE':'PUMA'})

df_acs['PUMA'] = df_acs['PUMA'].astype(str).apply('{:0>5}'.format)

df_fips1 = df_fips1[['State FIPS', 'County FIPS', 'County Name', 'MSA', 'MPO']]

df_acs2 = df_acs[df_acs['year'].isin(sequence(2022, 2031, 1))].merge(df_fips2[df_fips2['Years'] == '2022-2031'], on = ['State FIPS', 'PUMA'], how = 'left')
df_acs1 = df_acs[df_acs['year'].isin(sequence(2012, 2021, 1))].merge(df_fips2[df_fips2['Years'] == '2012-2021'], on = ['State FIPS', 'PUMA'], how = 'left')
df_acs = pd.concat([df_acs1, df_acs2])

df_acs = df_acs.merge(df_fips1, on = ['State FIPS', 'County FIPS'], how = 'left')

df_acs.head()

In [ ]:
df_mpo = df_acs[['State FIPS', 'MPO', 'year', 'SERIALNO', 'HHLDRRAC1P', 'housing_type', 'housing_burden', 'WGTP']].drop_duplicates()
df_mpo = df_mpo[~df_mpo.isna()]
print(df_mpo[(df_mpo['MPO'] == 'SACOG') & (df_mpo['year'] == 2021)].WGTP.sum())
df_mpo.head(10)

In [ ]:
df_mpo = df_acs[['State FIPS', 'MPO', 'year', 'SERIALNO', 'HHLDRRAC1P', 'housing_type', 'housing_burden', 'WGTP']].drop_duplicates()
df_mpo = df_mpo.groupby(['State FIPS', 'MPO', 'year', 'HHLDRRAC1P', 'housing_type', 'housing_burden'], as_index = False)['WGTP'].agg(sum)
print(df_mpo[(df_mpo['MPO'] == 'SACOG') & (df_mpo['year'] == 2021)].WGTP.sum())
df_mpo.head()

In [ ]:
df_mpo['Percentage'] = 100*df_mpo['WGTP']/df_mpo.groupby(['State FIPS', 'MPO', 'year', 'HHLDRRAC1P', 'housing_type'])['WGTP'].transform('sum')
df_mpo.head()

In [ ]:
geography = 'PUMA'

# Set output name for .xlsx files
name_output_long_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' Households.xlsx']
name_output_long_xlsx = "".join(name_output_long_xlsx)

# Set output name for .csv files
name_output_PUMA_csv = [indicator_name, '_PUMA_', estimate, '.csv']
name_output_PUMA_csv = "".join(name_output_PUMA_csv)

In [ ]:
report_theme = 'Vibrant and Inclusive Places'
sp_folder_out = 'Development\\Housing Cost'

# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out, indicator_name)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
df_mpo.to_excel(os.path.join(path_out_xlsx, name_output_long_xlsx), index = False)

print('')
print("Successfully exported")